# Millbrook NHS Trust — Python Intro

A short pandas / matplotlib / seaborn / scipy walkthrough of the Millbrook NHS Trust synthetic acute-care dataset. We'll cover:

1. Exploring `dim_patient` (and why it has more rows than patients).
2. Visualising the patient population by condition and deprivation.
3. Testing the 'Monday effect' in A&E arrivals — visually, then statistically.

**Full Python exercise set:** [github.com/leogodin217/nhs_sql_practice_data](https://github.com/leogodin217/nhs_sql_practice_data) (see `python_exercises.md`).

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats

DATA = '/kaggle/input/millbrook-nhs-acute-care-warehouse'
NHS_BLUE = '#005EB8'

## 1. Who are the patients?

`dim_patient` uses Type-2 Slowly Changing Dimensions. Each patient can have multiple rows as tracked attributes (status, admission count, total spell tariff) evolve. For a 'current' snapshot, filter to rows where `valid_to` is null.

In [ ]:
patients = pd.read_csv(f'{DATA}/dim_patient.csv')
current = patients[patients['valid_to'].isna()]

print(f"Total rows:       {len(patients):>6,}")
print(f"Current patients: {len(current):>6,}")
print(f"Unique IDs:       {current['id'].nunique():>6,}")
print()
print('Top primary conditions:')
print(current['primary_condition'].value_counts().head(10).to_string())

## 2. Visualise the population

Two quick views: primary condition mix, and the IMD (Index of Multiple Deprivation) decile distribution. NHS trusts serving urban catchments typically skew heavily toward deprived deciles.

In [ ]:
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(11, 4))

current['primary_condition'].value_counts().plot.barh(ax=ax1, color=NHS_BLUE)
ax1.set_xlabel('Patients')
ax1.set_title('By primary condition')
ax1.invert_yaxis()

current['imd_decile'].value_counts().sort_index().plot.bar(ax=ax2, color=NHS_BLUE)
ax2.set_xlabel('IMD decile (1 = most deprived)')
ax2.set_ylabel('Patients')
ax2.set_title('By deprivation')

plt.tight_layout()
plt.show()

## 3. The Monday effect in A&E

Group arrivals by calendar date, then look at the distribution of daily totals per day-of-week. A box plot reveals the shape — median, spread, outliers — that a bar chart of averages would hide.

In [ ]:
ed = pd.read_csv(f'{DATA}/fact_ed_arrival.csv')
ed['timestamp'] = pd.to_datetime(ed['timestamp'], format='ISO8601')
ed['day_name'] = ed['timestamp'].dt.day_name()
ed['date'] = ed['timestamp'].dt.date

daily = ed.groupby(['date', 'day_name']).size().reset_index(name='arrivals')

day_order = ['Monday', 'Tuesday', 'Wednesday', 'Thursday',
             'Friday', 'Saturday', 'Sunday']

fig, ax = plt.subplots(figsize=(9, 4))
sns.boxplot(data=daily, x='day_name', y='arrivals',
            order=day_order, color=NHS_BLUE, ax=ax)
ax.set_xlabel('Day of week')
ax.set_ylabel('Daily arrivals')
ax.set_title("A&E arrivals by day of week")
plt.xticks(rotation=20)
plt.tight_layout()
plt.show()

### Is it statistically significant?

A one-way ANOVA tests whether *any* day differs. Pairwise t-tests against Monday tell us *which* days differ, and in which direction.

In [ ]:
groups = [g['arrivals'].values for _, g in daily.groupby('day_name')]
f_stat, p = stats.f_oneway(*groups)

print(f"One-way ANOVA: F = {f_stat:.2f}, p = {p:.2e}")
print(f"Significant at 0.05? {'Yes' if p < 0.05 else 'No'}\n")

monday = daily.loc[daily['day_name'] == 'Monday', 'arrivals']
print('Pairwise t-tests, Monday vs. other days:')
for day in ['Tuesday', 'Wednesday', 'Thursday', 'Friday', 'Saturday', 'Sunday']:
    other = daily.loc[daily['day_name'] == day, 'arrivals']
    t, pv = stats.ttest_ind(monday, other)
    sig = '*' if pv < 0.05 else ' '
    print(f"  Monday vs {day:9s}  t = {t:6.2f}   p = {pv:.4f} {sig}")

scipy turns *'it looks like Monday is busier'* into *'Monday averages N more arrivals, p < 0.001'*. That's the difference between an observation and evidence.

## Where to next

The companion repo has deeper exercises: length-of-stay distribution, deprivation-vs-outcome correlation, and a logistic-regression readmission model with scikit-learn.

Repo: [github.com/leogodin217/nhs_sql_practice_data](https://github.com/leogodin217/nhs_sql_practice_data)